# CNN-EXP-005 — STEP++ exp

Меняем только выход модели: общий backbone из CNN-EXP-001 получает две головы.
Presence предсказывает `count > 0`, intensity будет предсказывать `log1p(count)` на положительных позициях.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import torch
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)

Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))

DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])

assert train_contigs.isdisjoint(validation_contigs)

display(split.summary)
print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3


In [4]:
profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(batch_size=4)

print("Обычный вход:", train_batch.x.shape)
print("Обе ориентации:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Положительных:", int(train_batch.target.sum()))
print("Sampling branches:", train_batch.sampling_branch)

Обычный вход: torch.Size([4, 5, 1284])
Обе ориентации: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Положительных: 33
Sampling branches: ('negative', 'positive', 'positive', 'negative')


## Двухголовая модель

Backbone полностью совпадает с CNN-EXP-001. Добавляется только второй выходной канал.

In [5]:
multitask_model = create_cnn_presence_intensity().to(device)
multitask_model.eval()

x_both_gpu = train_batch.x_both_strands.to(device)

with torch.no_grad():
    presence_logits, intensity_prediction = multitask_model(x_both_gpu)

parameter_count = sum(
    parameter.numel()
    for parameter in multitask_model.parameters()
    if parameter.requires_grad
)

expected_shape = (
    2 * len(train_batch.tiles),
    1,
    profile_config.input_length,
)

print("Общий вход:", x_both_gpu.shape, x_both_gpu.device)
print("Presence:", presence_logits.shape, presence_logits.device)
print("Intensity:", intensity_prediction.shape, intensity_prediction.device)
print("Параметров:", parameter_count)

assert presence_logits.shape == expected_shape
assert intensity_prediction.shape == expected_shape
assert parameter_count == 59138

Общий вход: torch.Size([8, 5, 1284]) privateuseone:0
Presence: torch.Size([8, 1, 1284]) privateuseone:0
Intensity: torch.Size([8, 1, 1284]) privateuseone:0
Параметров: 59138


In [6]:
multitask_model.eval()

with torch.no_grad():
    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

print("Plus presence:", plus_presence.shape)
print("Minus presence reverse:", minus_presence_reverse.shape)
print("Plus intensity:", plus_intensity.shape)
print("Minus intensity reverse:", minus_intensity_reverse.shape)

assert plus_presence.shape == (4, 1, 1024)
assert minus_presence_reverse.shape == (4, 1, 1024)
assert plus_intensity.shape == (4, 1, 1024)
assert minus_intensity_reverse.shape == (4, 1, 1024)

Plus presence: torch.Size([4, 1, 1024])
Minus presence reverse: torch.Size([4, 1, 1024])
Plus intensity: torch.Size([4, 1, 1024])
Minus intensity reverse: torch.Size([4, 1, 1024])


In [7]:
plus_count = (
    train_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    train_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target = torch.log1p(
    plus_count
)

minus_intensity_target_reverse = torch.log1p(
    minus_count_reverse
)

plus_intensity_mask = (
    plus_count > 0
)

minus_intensity_mask_reverse = (
    minus_count_reverse > 0
)

print("Plus count:", plus_count.shape)
print("Minus count reverse:", minus_count_reverse.shape)

print(
    "Положительных plus:",
    int(plus_intensity_mask.sum()),
)

print(
    "Положительных minus:",
    int(minus_intensity_mask_reverse.sum()),
)

print(
    "Диапазон intensity target:",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).min()
    ),
    "—",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).max()
    ),
)

assert torch.equal(
    plus_intensity_mask.float(),
    train_batch.plus_target,
)

assert torch.equal(
    minus_intensity_mask_reverse.float(),
    train_batch.minus_target_reverse,
)

Plus count: torch.Size([4, 1, 1024])
Minus count reverse: torch.Size([4, 1, 1024])
Положительных plus: 32
Положительных minus: 1
Диапазон intensity target: 0.6931471824645996 — 4.143134593963623


In [8]:
import warnings


INTENSITY_LOSS_WEIGHT = 0.1

multitask_model.train()

optimizer = DirectMLAdam(
    multitask_model.parameters(),
    lr=1e-3,
)

# Presence targets и веса.
plus_target_gpu = (
    train_batch.plus_target.to(device)
)
minus_target_reverse_gpu = (
    train_batch.minus_target_reverse.to(device)
)

plus_presence_weight_gpu = (
    train_batch.plus_loss_weight.to(device)
)
minus_presence_weight_reverse_gpu = (
    train_batch.minus_loss_weight_reverse.to(device)
)

# Intensity targets.
plus_intensity_target_gpu = (
    plus_intensity_target.to(device)
)
minus_intensity_target_reverse_gpu = (
    minus_intensity_target_reverse.to(device)
)

# Положительные веса в BCE нормированы на 0.5.
# Умножение на 2 превращает их в условное распределение
# только среди положительных position-strand.
plus_intensity_weight_gpu = (
    2.0
    * train_batch.plus_loss_weight
    * plus_intensity_mask.float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * train_batch.minus_loss_weight_reverse
    * minus_intensity_mask_reverse.float()
).to(device)

optimizer.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )

    total_loss.backward()
    optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = (
    multitask_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .item()
)

intensity_gradient = (
    multitask_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .item()
)

print("Presence loss:", presence_loss.item())
print("Intensity loss:", intensity_loss.item())
print("Total loss:", total_loss.item())
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

Presence loss: 0.7468885183334351
Intensity loss: 0.5407075881958008
Total loss: 0.8009592890739441
Presence gradient: 2.4733827114105225
Intensity gradient: 0.7240514159202576
CPU fallback: []


In [9]:
import math
import time
import warnings


PILOT_STEPS = 200
PILOT_BATCH_SIZE = 4
PILOT_SEED = 42
LOG_EVERY = 20
INTENSITY_LOSS_WEIGHT = 0.1

torch.manual_seed(PILOT_SEED)

pilot_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=PILOT_SEED,
)

pilot_model = (
    create_cnn_presence_intensity()
    .to(device)
)

pilot_optimizer = DirectMLAdam(
    pilot_model.parameters(),
    lr=1e-3,
)

pilot_model.train()

pilot_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, PILOT_STEPS + 1):
        batch = pilot_generator.sample_batch(
            batch_size=PILOT_BATCH_SIZE
        )

        # Вход обеих ориентаций уже объединён на CPU.
        x_gpu = batch.x_both_strands.to(device)

        # Presence targets.
        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Intensity targets готовим и разворачиваем на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        pilot_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            pilot_model,
            x_gpu,
            batch_size=PILOT_BATCH_SIZE,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT * intensity_loss
        )

        if not math.isfinite(total_loss.item()):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        pilot_optimizer.step()

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.95 * ema_presence
                + 0.05 * presence_value
            )
            ema_intensity = (
                0.95 * ema_intensity
                + 0.05 * intensity_value
            )
            ema_total = (
                0.95 * ema_total
                + 0.05 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        pilot_history.append(
            {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
            }
        )

        if step == 1 or step % LOG_EVERY == 0:
            elapsed = (
                time.perf_counter()
                - started_at
            )

            print(
                f"step={step:3d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

print()
print("Время:", round(elapsed, 2), "сек")
print("Финальный EMA presence:", round(ema_presence, 6))
print("Финальный EMA intensity:", round(ema_intensity, 6))
print("Финальный EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)

assert not fallback_warnings

step=  1 presence=0.8008 intensity=1.0298 total=0.9038 ema=(0.8008, 1.0298, 0.9038) positions/s=146,426
step= 20 presence=0.5853 intensity=0.5705 total=0.6423 ema=(0.7237, 0.6276, 0.7865) positions/s=153,887
step= 40 presence=0.5992 intensity=0.4806 total=0.6473 ema=(0.6830, 0.5075, 0.7338) positions/s=166,327
step= 60 presence=0.5265 intensity=0.3868 total=0.5652 ema=(0.6446, 0.4388, 0.6885) positions/s=169,129
step= 80 presence=0.5636 intensity=0.3210 total=0.5957 ema=(0.5948, 0.3473, 0.6295) positions/s=171,250
step=100 presence=0.6649 intensity=1.3155 total=0.7964 ema=(0.6067, 0.3915, 0.6458) positions/s=174,426
step=120 presence=0.4328 intensity=0.0687 total=0.4397 ema=(0.5551, 0.3329, 0.5884) positions/s=175,991
step=140 presence=0.6263 intensity=0.7438 total=0.7006 ema=(0.5557, 0.2790, 0.5836) positions/s=177,782
step=160 presence=0.3971 intensity=0.1314 total=0.4102 ema=(0.5648, 0.2730, 0.5921) positions/s=179,403
step=180 presence=0.4761 intensity=0.2394 total=0.5001 ema=(0.51

In [10]:
from pathlib import Path


EXP005_SEED = 42
EXP005_STEPS = 10_0000
EXP005_BATCH_SIZE = 4
EXP005_LOG_EVERY = 250
EXP005_CHECKPOINT_STEPS = {
    2_000,
    5_000,
    10_000,
}
INTENSITY_LOSS_WEIGHT = 0.1

EXP005_DIR = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP005"
    / "run_001"
)

EXP005_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.manual_seed(EXP005_SEED)

exp002_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=EXP005_SEED,
)

exp002_model = (
    create_cnn_presence_intensity()
    .to(device)
)

exp002_optimizer = DirectMLAdam(
    exp002_model.parameters(),
    lr=1e-3,
)

print("Артефакты:", EXP005_DIR)
print("Параметров:", sum(
    parameter.numel()
    for parameter in exp002_model.parameters()
    if parameter.requires_grad
))
print("Шагов:", EXP005_STEPS)
print("Intensity weight:", INTENSITY_LOSS_WEIGHT)

Артефакты: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP005\run_001
Параметров: 59138
Шагов: 100000
Intensity weight: 0.1


In [11]:
import copy
import math
import time
import warnings

import pandas as pd


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def save_exp002_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": "CNN-EXP-005",
        "model_type": "presence_intensity",
        "step": step,
        "model_state_dict": model_state_on_cpu(
            exp002_model
        ),
        "optimizer_state_dict": tree_to_cpu(
            exp002_optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            exp002_generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },
        "training_config": {
            "seed": EXP005_SEED,
            "steps": EXP005_STEPS,
            "batch_size": EXP005_BATCH_SIZE,
            "learning_rate": 1e-3,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },
        "seen_position_strand": (
            seen_position_strand
        ),
        "elapsed_seconds": elapsed_seconds,
        "history": list(history),
    }

    torch.save(checkpoint, path)


exp002_model.train()

exp002_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, EXP005_STEPS + 1):
        batch = exp002_generator.sample_batch(
            batch_size=EXP005_BATCH_SIZE
        )

        x_gpu = batch.x_both_strands.to(device)

        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Всё выравнивание minus выполняется на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        exp002_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            exp002_model,
            x_gpu,
            batch_size=EXP005_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        exp002_optimizer.step()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )
            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )
            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % EXP005_LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            record = {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
                "elapsed_seconds": elapsed,
                "seen_position_strand": (
                    seen_position_strand
                ),
                "positions_per_second": (
                    seen_position_strand / elapsed
                ),
            }

            exp002_history.append(record)

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

        if step in EXP005_CHECKPOINT_STEPS:
            checkpoint_path = (
                EXP005_DIR
                / f"model_step_{step:05d}.pt"
            )

            save_exp002_checkpoint(
                checkpoint_path,
                step=step,
                elapsed_seconds=(
                    time.perf_counter()
                    - started_at
                ),
                seen_position_strand=(
                    seen_position_strand
                ),
                history=exp002_history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

final_checkpoint_path = (
    EXP005_DIR / "exp002_final.pt"
)
history_path = (
    EXP005_DIR / "training_history.csv"
)

save_exp002_checkpoint(
    final_checkpoint_path,
    step=EXP005_STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=exp002_history,
)

pd.DataFrame(exp002_history).to_csv(
    history_path,
    index=False,
)

print()
print("Общее время:", round(elapsed, 2), "сек")
print(
    "Время шага:",
    round(elapsed / EXP005_STEPS, 4),
    "сек",
)
print(
    "Увидено position-strand:",
    seen_position_strand,
)
print(
    "Финальный EMA presence:",
    round(ema_presence, 6),
)
print(
    "Финальный EMA intensity:",
    round(ema_intensity, 6),
)
print(
    "Финальный EMA total:",
    round(ema_total, 6),
)
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", final_checkpoint_path)
print("История:", history_path)

assert not fallback_warnings
assert final_checkpoint_path.is_file()
assert history_path.is_file()

step=    1 presence=0.8008 intensity=1.0298 total=0.9038 ema=(0.8008, 1.0298, 0.9038) positions/s=135,723
step=  250 presence=0.4711 intensity=0.4628 total=0.5174 ema=(0.5688, 0.3784, 0.6067) positions/s=179,244
step=  500 presence=0.5533 intensity=0.0413 total=0.5574 ema=(0.5006, 0.2447, 0.5251) positions/s=178,883
step=  750 presence=0.5593 intensity=0.2969 total=0.5890 ema=(0.4893, 0.2504, 0.5144) positions/s=179,198
step= 1000 presence=0.3140 intensity=0.2184 total=0.3358 ema=(0.4803, 0.2480, 0.5051) positions/s=178,798
step= 1250 presence=0.2984 intensity=0.8980 total=0.3882 ema=(0.4747, 0.2463, 0.4994) positions/s=179,293
step= 1500 presence=0.3753 intensity=0.2035 total=0.3957 ema=(0.4571, 0.2211, 0.4792) positions/s=179,418
step= 1750 presence=0.6384 intensity=0.0877 total=0.6472 ema=(0.4436, 0.2536, 0.4690) positions/s=179,403
step= 2000 presence=0.2949 intensity=0.3069 total=0.3255 ema=(0.4355, 0.2650, 0.4620) positions/s=179,211
step= 2250 presence=0.4177 intensity=0.0605 to

In [12]:
from ml_project.epic_cnn.evaluation import (
    evaluate_presence_intensity_model,
    load_cnn_presence_intensity_checkpoint,
)


EXP005_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP005"
    / "run_001"
    / "exp002_final.pt"
)

loaded_exp002 = (
    load_cnn_presence_intensity_checkpoint(
        EXP005_CHECKPOINT_PATH,
        device=device,
    )
)

validation_model = loaded_exp002.model
validation_config = loaded_exp002.profile_config
checkpoint = loaded_exp002.checkpoint

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=validation_config,
    seed=42,
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_generator.tile_index))
print(
    "Validation position-strand:",
    int(
        validation_generator
        .tile_index["allowed_position_strand"]
        .sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)
print(
    "Device:",
    next(validation_model.parameters()).device,
)

Checkpoint step: 100000
Validation tiles: 47743
Validation position-strand: 83807486
Validation positives: 56205
Device: privateuseone:0


In [13]:
validation_result = (
    evaluate_presence_intensity_model(
        validation_model,
        validation_generator,
        device=device,
        batch_size=8,
        score_decimals=5,
        progress_every_batches=500,
    )
)

tiles=     8/47743 positions=    11484 positions/s=928,007
tiles=  4000/47743 positions=  6861742 positions/s=1,935,636
tiles=  8000/47743 positions= 14166170 positions/s=2,053,174
tiles= 12000/47743 positions= 21458144 positions/s=2,077,547
tiles= 16000/47743 positions= 28627148 positions/s=2,094,431
tiles= 20000/47743 positions= 35584246 positions/s=2,092,134
tiles= 24000/47743 positions= 42571942 positions/s=2,083,752
tiles= 28000/47743 positions= 49714988 positions/s=2,092,553
tiles= 32000/47743 positions= 56582858 positions/s=2,078,096
tiles= 36000/47743 positions= 63201944 positions/s=2,058,006
tiles= 40000/47743 positions= 70164960 positions/s=2,061,917
tiles= 44000/47743 positions= 77380732 positions/s=2,071,854
tiles= 47743/47743 positions= 83807486 positions/s=2,065,604


In [14]:
VALIDATION_METRICS_PATH = (
    EXP005_CHECKPOINT_PATH.parent
    / "validation_metrics.csv"
)

validation_result.metrics.to_csv(
    VALIDATION_METRICS_PATH,
    index=False,
)

overall_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        == "overall"
    ]
    .reset_index(drop=True)
)

display(overall_metrics)

print(
    "Validation time:",
    round(validation_result.elapsed_seconds, 2),
    "sec",
)
print(
    "Position-strand/s:",
    f"{validation_result.positions_per_second:,.0f}",
)
print(
    "CPU fallback:",
    list(validation_result.fallback_warnings),
)
print("Metrics:", VALIDATION_METRICS_PATH)

,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,presence_probability,overall,83807486,56205,0.000671,0.072003,107.364413,0.190218,0.149902
1,intensity_prediction,overall,83807486,56205,0.000671,0.040791,60.823100,0.207464,0.228318
2,expected_signal,overall,83807486,56205,0.000671,0.051214,76.364913,0.208714,0.205293


Validation time: 40.68 sec
Position-strand/s: 2,060,151
CPU fallback: []
Metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP005\run_001\validation_metrics.csv


In [15]:
per_contig_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        != "overall"
    ]
    .sort_values(
        ["segment", "score"]
    )
    .reset_index(drop=True)
)

display(per_contig_metrics)

,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,expected_signal,NC_064034.1,33949114,23529,0.000693,0.055117,79.526477,0.231525,0.229842
1,intensity_prediction,NC_064034.1,33949114,23529,0.000693,0.044174,63.736357,0.227137,0.236417
2,presence_probability,NC_064034.1,33949114,23529,0.000693,0.072900,105.183963,0.211544,0.186635
3,expected_signal,NC_064041.1,24146754,13984,0.000579,0.045131,77.929442,0.251954,0.250490
4,intensity_prediction,NC_064041.1,24146754,13984,0.000579,0.035908,62.004465,0.251655,0.260611
5,presence_probability,NC_064041.1,24146754,13984,0.000579,0.065527,113.148181,0.235074,0.207998
6,expected_signal,NC_064042.1,25711618,18692,0.000727,0.051912,71.406848,0.241365,0.241318
7,intensity_prediction,NC_064042.1,25711618,18692,0.000727,0.041085,56.514303,0.241331,0.254208
8,presence_probability,NC_064042.1,25711618,18692,0.000727,0.076384,105.069816,0.220902,0.190625


In [16]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "RF++ охват очень сильно влияет нужно смотреть шире!"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-005',
        title='RF++',
        hypothesis='увеличиваю ещё сильнее RF до максимума',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP005_rf.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP005/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-003',
        reference_metrics='artifacts/experiments/CNN_EXP003/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

Obsidian: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-005.md
Summary: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP005\run_001\experiment_summary.json
Plots: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP005\run_001\plots


In [18]:
# Запускай эту ячейку только после обучения и полной validation.
SAVE_EXPERIMENT = True
FINAL_DECISION = "adopt"  # review / adopt / reject / iterate
FINAL_NOTES = "увеличенное число шагов сильно улучшает AP"

if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import finalize_cnn_experiment

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id='CNN-EXP-005',
        title='STEP++ exp',
        hypothesis='увеличение шагов даст  явный прирост AP',
        notebook_path='notebooks/experiments/CNN_experiments/CNN_EXP005_step_exp.ipynb',
        artifact_dir=PROJECT_ROOT / 'artifacts/experiments/CNN_EXP005/run_001',
        training_history=history_path,
        candidate_metrics=VALIDATION_METRICS_PATH,
        reference_experiment='CNN-EXP-004',
        reference_metrics='artifacts/experiments/CNN_EXP004/run_001/validation_metrics.csv',
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )
    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)
else:
    print("Сохранение выключено. Установи SAVE_EXPERIMENT = True после validation.")

Obsidian: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-005.md
Summary: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP005\run_001\experiment_summary.json
Plots: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP005\run_001\plots
